# SpaceX Falcon 9 - Interactive Visual Analytics with Folium

Mapping out the Falcon 9 launch sites and each individual launch outcome (success/failure) geographically, to see if location patterns relate to landing success.

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

df = pd.read_csv('spacex_launch_geo.csv')
launch_sites_df = df.groupby('Launch Site').first().reset_index()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610746


### Mark all launch sites on a map

In [2]:
site_map = folium.Map(location=[29.5, -95], zoom_start=4.5)

for _, row in launch_sites_df.iterrows():
    folium.Circle(
        [row['Lat'], row['Long']], radius=1000, color='#3186cc', fill=True
    ).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(
        [row['Lat'], row['Long']],
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % row['Launch Site'])
    ).add_to(site_map)

site_map

All four launch sites sit right on the coast — two in Florida (Cape Canaveral / KSC) and one in California (Vandenberg) — which makes sense, since rockets are launched over open water so that any debris from a failure falls into the ocean rather than over land.

### Mark every individual launch, colored by outcome

In [3]:
marker_cluster = MarkerCluster().add_to(site_map)

df['marker_color'] = df['class'].apply(lambda x: 'green' if x == 1 else 'red')

for _, row in df.iterrows():
    folium.Marker(
        location=[row['Lat'], row['Long']],
        icon=folium.Icon(color='white', icon_color=row['marker_color']),
        popup=row['Launch Site']
    ).add_to(marker_cluster)

site_map

Zooming into any site's cluster shows the mix of green (successful landing) and red (unsuccessful) markers — KSC LC-39A and the newer CCAFS SLC-40 records lean noticeably more green than the earlier CCAFS LC-40 launches, again reflecting improvement over time.

### Distance from a launch site to its nearest coastline

In [4]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Example: CCAFS SLC-40 to its closest coastline point
launch_site_lat, launch_site_lon = 28.563197, -80.576820
coastline_lat, coastline_lon = 28.56367, -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

coordinate = [coastline_lat, coastline_lon]
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html='<div style="font-size: 12px; color:#d35400;"><b>%.2f KM</b></div>' % distance_coastline)
)
site_map.add_child(distance_marker)

lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], coordinate], weight=1)
site_map.add_child(lines)

print("Distance from CCAFS SLC-40 to the nearest coastline: {:.2f} km".format(distance_coastline))
site_map

Distance from CCAFS SLC-40 to the nearest coastline: 0.51 km


Launch sites sit less than a kilometer from the coastline, confirming the pattern above — SpaceX (like most orbital launch providers) deliberately builds pads as close to open water as the site allows.

## Summary

Geographically, launch site placement follows a very deliberate pattern: right on the coast, to keep the flight path over water. The marker clustering also visually reinforces the same trend seen in the earlier EDA — later launches (concentrated at KSC LC-39A and the upgraded CCAFS SLC-40) cluster with far more green (successful) markers than the earliest CCAFS LC-40 launches.